<h1 style="color:orange">EXERCISE CLASS 4 </h1>

## EXERCISE 1

A chemical plant operates several reactors with low temperature solvents, where temperature fluctuations can affect reaction rates, product yield, and safety. To ensure process stability, the quality control team conducts regular temperature sampling throughout the facility. At fixed time intervals, temperature readings are taken from four randomly selected reactors or processing units. The collected data in °C is reported in `reactors.csv`.

1. Design a control chart for the mean. 
2. Consider the samples as individual measurements. Design a control chart for the mean.


### 1. Design a control chart for the mean. 

In [ ]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import qdatoolkit as qda

# Import the dataset
data = pd.read_csv('../Dataset/reactors.csv')

# Inspect the dataset
data.head()

Inspect the data by plotting individual data points.

In [ ]:
# Make a scatter plot of all the columns against the index
plt.plot(data['x1'], linestyle='none', marker='o', label = 'x1')
plt.plot(data['x2'], linestyle='none', marker='o', label = 'x2')
plt.plot(data['x3'], linestyle='none', marker='o', label = 'x3')
plt.plot(data['x4'], linestyle='none', marker='o', label = 'x4')
# place the legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
plt.show()

Looks like outliers are present, or - more likely - the distribution is skewed.

In [ ]:
# Stack the data into a single column
data_stack = data.stack()

# Plot a histogram of the data_stack
data_stack.hist()
plt.show()

> The distribution looks skewed. Let's verify the normality assumption.

In [ ]:
# We can use the qda.Assumptions().normality() method that incorporates a Shapiro-Wilk test and the QQ-plot of the data

_ = qda.Assumptions(data_stack).normality()

The data are non-normal. Therefore, we cannot use the X-bar and R chart on the raw data. We need to transform the data.

But what happens if we neglect the normality violation and use the X-bar and R chart on the raw data?

In [ ]:
# X-bar and R charts
data_XR = qda.ControlCharts.XbarR(data)

> OOC observations may be due to a violation of control chart assumptions...

> Let's transform the data to make it more normal using the Box-Cox transformation.
>
> Remember the Box-Cox transformation is defined as:
> $$x_{BC,i} = \left\{ \begin{array}{ll} \frac{x_i^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln x_i & \text{if } \lambda = 0 \end{array} \right.$$

In [ ]:
# Box-Cox transformation and return the transformed data
[data_BC, lmbda] = stats.boxcox(data_stack)

print('Lambda = %.3f' % lmbda)

# Plot a histogram of the transformed data
plt.hist(data_BC)
plt.show()

In [ ]:
# It is also possible to find the best value of lambda for the transformation
fig = plt.figure()
ax = fig.add_subplot(111)
stats.boxcox_normplot(data_stack, -2, 2, plot=ax)
# add grid
ax.grid(True)

> By default, the Box-Cox function used Lambda = -0.201. A more interpretable (and still close to optimum) value is Lambda = 0.

In [ ]:
# Use lambda = 0 for Box-Cox transformation and return the transformed data
data_BC_lambda0 = stats.boxcox(data_stack, lmbda=0)

# Plot a histogram of the transformed data
plt.hist(data_BC_lambda0)
plt.show()

> Now the data seem to follow a normal distribution. Let's verify this by testing the normality.

In [ ]:
_ = qda.Assumptions(data_BC_lambda0).normality()

> Normality is verified. We can now use the X-bar and R chart on the transformed data.

In [ ]:
# First we need to unstack the data
data_BC_unstack = data_BC_lambda0.reshape(data.shape)
# and convert it to a DataFrame
data_BC_unstack = pd.DataFrame(data_BC_unstack, columns = data.columns)

# Print out the transformed data
data_BC_unstack.head()

In [ ]:
# X-bar and R charts
data_BC_XR = qda.ControlCharts.XbarR(data_BC_unstack)

 > Even if the normality assumption holds, the control limits of the Xbar chart look too narrow with respect to the natural variability of the statistic. This can be caused by a violation of assumptions (independence) within the sample. Thus, the Xbar-R control chart may be not appropriate to monitor these data. 

### 2. Consider the samples as individual measurements. Design a control chart for the mean.

> Design a "Between groups" control chart, i.e., a chart that assumes all the samples to be individual measurements.

In [ ]:
# Create a new dataframe that stores the mean of all the samples
data_Xbar_BC = pd.DataFrame(data_BC_XR['sample_mean'])

# Build the IMR chart using this new dataframe
data_Xbar = qda.ControlCharts.IMR(data_Xbar_BC, 'sample_mean')

 > With the I chart we can get rid of the violation of the independence assumption within the sample. The MR chart allows monitoring the between sample variability, while the R chart designed before can still be used to monitor the within sample variability. 
A control charting scheme quite effective in this case is the so-called I-MR-R control chart (or I-MR-S, if the S chart is used in place of the R chart), where: 1) the I chart allows monitoring the mean of the process treating sample means as individual observations; 2) the MR chart allows monitoring the between sample variability; 3) the MR chart allows monitoring the within sample variability.

In [ ]:
# Design a I-MR-R control chart

# Build the IMR chart using this new dataframe
data_Xbar = qda.ControlCharts.IMR(data_Xbar, 'sample_mean')

# Plot the R chart as well
plt.title('R chart')
plt.plot(data_BC_XR['sample_range'], color='b', linestyle='--', marker='o')
plt.plot(data_BC_XR['R_UCL'], color='r')
plt.plot(data_BC_XR['R_CL'], color='g')
plt.plot(data_BC_XR['R_LCL'], color='r')
plt.ylabel('Sample range')
plt.xlabel('Sample number')
# add the values of the control limits on the right side of the plot
plt.text(len(data_BC_XR)+.5, data_BC_XR['R_UCL'].iloc[0], 'UCL = {:.3f}'.format(data_BC_XR['R_UCL'].iloc[0]), verticalalignment='center')
plt.text(len(data_BC_XR)+.5, data_BC_XR['R_CL'].iloc[0], 'CL = {:.3f}'.format(data_BC_XR['R_CL'].iloc[0]), verticalalignment='center')
plt.text(len(data_BC_XR)+.5, data_BC_XR['R_LCL'].iloc[0], 'LCL = {:.3f}'.format(data_BC_XR['R_LCL'].iloc[0]), verticalalignment='center')
# highlight the points that violate the alarm rules
plt.plot(data_BC_XR['R_TEST1'], linestyle='none', marker='s', color='r', markersize=10)
plt.show()

 > There is an alarm in the 13th sample. A search for assignable causes shall be performed. In the absence of information about possible assignable causes, the alarm can be labelled as a false alarm, and the design phase of the control chart is over.